In [7]:
import torch
from torch import nn
from torchvision import models, datasets, transforms
from tqdm import tqdm

In [8]:
base_model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
base_model.heads = nn.Linear(in_features=768, out_features=100)
base_model = base_model.to(device)

In [9]:
cifar100_dataset_train = datasets.CIFAR100(root='./data', train=True, download=True, transform=transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
]))
cifar100_train_loader = torch.utils.data.DataLoader(cifar100_dataset_train, batch_size=8, shuffle=True, num_workers=4)

cifar100_dataset_test = datasets.CIFAR100(root='./data', train=False, download=True, transform=transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
]))
cifar100_test_loader = torch.utils.data.DataLoader(cifar100_dataset_test, batch_size=8, shuffle=False, num_workers=4)

In [10]:
def compute_accuracy(model):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in tqdm(cifar100_test_loader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

In [ ]:
def train_model(model, epochs=3):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    model.train()
    for epoch in range(epochs):
        pbar = tqdm(cifar100_train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=True)
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')
    return model

: 

In [ ]:
train_model(base_model, epochs=3)

Epoch 1/3:   1%|          | 62/6250 [00:26<44:31,  2.32it/s, loss=4.6223]  